# Laboratório integrado — Detecção de mixers Ethereum, auditorias e gráficos

**Interface única para estudar e demonstrar EXP-01J, EXP-02A e EXP-07.** Importa os resultados oficiais congelados do Google Drive, verifica hashes, explica a validade e cria gráficos em alta resolução. Os notebooks originais continuam preservados para a reprodução detalhada dos EXP-01 a EXP-07. **Executar Tudo** neste painel não equivale a reprocessar todos os dumps da Ethereum.

**Alertas não são rótulos.** Em 2026 foram nove alertas (três Tornado conhecidos e seis indeterminados); há um único contrato positivo nas 23 linhas do treino. Não se estima precision ou F1 populacional para a rede.

**Inspiração visual:** fluxos de metodologia semelhantes ao estudo Leale e Sendin (SBSeg 2025); diagramas de pipeline e visualização de padrões comparáveis em estilo (não em métricas nem escopo) aos artigos GNN-based Detection... e MixGuard. Nada foi copiado das figuras originais.

In [ ]:
%pip -q install pandas numpy matplotlib scikit-learn pyarrow google-api-python-client

## 1. Carregar entradas científicas e conferir integridade

Usa a autorização Google Drive do próprio Colab. Não inserir tokens pessoais, não disponibilizar as planilhas cegas EXP-05 e não modificar os dados de entrada.

In [ ]:
from pathlib import Path
import os, io, json, hashlib, pandas as pd, numpy as np, matplotlib.pyplot as plt
from google.colab import auth
from googleapiclient.discovery import build
from googleapiclient.http import MediaIoBaseDownload,MediaFileUpload
from IPython.display import display
from datetime import datetime,timezone

auth.authenticate_user()
service=build('drive','v3')
ROOT=Path('/content/lab_ethereum');ROOT.mkdir(exist_ok=True)
FIG=ROOT/'figures';FIG.mkdir(exist_ok=True)
IDS={
 'manifesto_exp07.json':'1rAwtsQbsX5rhmgI-OLn8GBmJxy0rgGAo',
 'auditoria_matriz.json':'1Ghgugyo8U0Mhn4GAsZlaOmnJ5Clq4nUe',
 'features_treino_original.parquet':'1_Sry0K0JusRL3PcxEweyC_9hnaP1vdRy',
 'folds_congelados.json':'1cg1fkkmFudgY-eqej__5euY_gI5Yac_g',
 'metricas_12_combinacoes_por_fold.csv':'1TB1oy9ZwzYL_EMggYTvZqY3ENHXZj0Uh',
 'resumo_12_combinacoes.csv':'1h8dC2pmki_TteMGXKLHaVAEmVK8duKTO',
 'sinalizados_teste_2026.csv':'11kWohs3VvdDGVjwsN_t-S8EQVix6n9mf',
}
def sha(p):return hashlib.sha256(Path(p).read_bytes()).hexdigest()
def obter(name,id):
    path=ROOT/name
    if path.exists():return path
    req=service.files().get_media(fileId=id)
    b=io.BytesIO();down=MediaIoBaseDownload(b,req);done=False
    while not done:_,done=down.next_chunk()
    path.write_bytes(b.getvalue());return path
for fn,fid in IDS.items():obter(fn,fid)
manifesto=json.loads((ROOT/'manifesto_exp07.json').read_text())
checks={f:sha(ROOT/f)==expected for f,expected in manifesto['arquivos_sha256'].items()}
assert all(checks.values()),f'Integridade EXP-07 divergente: {checks}'
print('Integridade oficial EXP-07 OK:',len(checks),'arquivos SHA-256 conferidos')
print('Fonte da amostra janeiro 2026: arquivo oficial Drive; não é ground truth externo')
AUD=json.loads((ROOT/'auditoria_matriz.json').read_text())
print('Auditoria da matriz:',AUD['class_counts'],'contratos por classe:',AUD['contracts_by_class'])
assert AUD['contracts_by_class'].get('1')==1

## 2. Figuras de reprodução e sensibilidade (contagens auditadas)

As contagens 63, 9, 49, 14, 9 e 7 vêm da reprodução final e do protocolo EXP-01J. Os dois tipos de gráfico mostram **quantos alertas** foram produzidos, não quantos são corretos. A matriz CSV com resumo EXP-07 é baixada e verificável por hash.

In [ ]:
plt.rcParams.update({'figure.figsize':(8.5,4.5),'figure.dpi':120,'savefig.dpi':250,'font.size':11,'axes.spines.top':False,'axes.spines.right':False})
PETROL='#176B78';GRAY='#64748B';ORANGE='#AA5935'
def export(nome):
    plt.tight_layout();p=FIG/nome;plt.savefig(p,bbox_inches='tight');plt.show();print('Figura salva:',p)
fig,ax=plt.subplots()
labels=['Artigo SBSeg 2025','Reprodução consolidada'];vals=[63,9]
ax.barh(labels[::-1],vals[::-1],color=[PETROL,GRAY],height=.56)
for y,v in enumerate(vals[::-1]):ax.text(v+.8,y,str(v),va='center',fontweight='bold')
ax.set(xlim=(0,73),xlabel='Quantidade de alertas — 29/10/2020',title='Reprodução: diferença na contagem de alertas')
ax.text(.99,.02,'Pools Tornado Cash identificados: 3/3 em ambos',transform=ax.transAxes,ha='right',fontsize=10)
export('01_comparacao_alertas_2020.png')

fig,ax=plt.subplots();xs=[1,5,10,20];ys=[49,14,9,7]
ax.plot(xs,ys,'o-',color=PETROL,linewidth=2.5,markersize=8)
for x,y in zip(xs,ys):ax.annotate(str(y),(x,y),xytext=(0,8),textcoords='offset points',ha='center',fontweight='bold')
ax.set(xticks=xs,xlabel='Mínimo de transações por endereço',ylabel='Total de alertas',title='EXP-01J — Sensibilidade do filtro min_tx')
ax.grid(alpha=.2);export('02_sensibilidade_min_tx.png')

## 3. Mapa de calor dos modelos e ablação EXP-07

F1 **interno** em cinco folds **por linhas**. O mesmo contrato positivo aparece tanto no treino como na validação em todos os cinco folds. Não chame isso de detecção de mixers inéditos.

In [ ]:
SUM=pd.read_csv(ROOT/'resumo_12_combinacoes.csv')
order=['RandomForest','DecisionTree','GradientBoosting','RegressaoLogistica']
sets=['A_original_5','B_sem_volume_4','C_entropias_3']
M=SUM.pivot(index='modelo',columns='configuracao',values='f1_mean').reindex(index=order,columns=sets)
assert M.notna().all().all() and M.shape==(4,3)
fig,ax=plt.subplots(figsize=(8.7,4.8));im=ax.imshow(M.values,cmap='YlGnBu',vmin=.45,vmax=1,aspect='auto')
ax.set_xticks(range(3),['5 atributos','Sem volume','Apenas entropias'])
ax.set_yticks(range(4),['Random Forest','Decision Tree','Gradient Boosting','Regressão Logística'])
for r in range(4):
 for c in range(3):
    val=M.values[r,c];ax.text(c,r,f'{val:.3f}',ha='center',va='center',fontweight='bold',color='white' if val>.84 else 'black')
fig.colorbar(im,ax=ax,label='F1 médio interno');ax.set_title('EXP-07 — 4 modelos × 3 grupos de características')
export('03_heatmap_f1_interno.png')

## 4. Distribuição das características e dispersão 2D do treino

As figuras abaixo mostram a **matriz original** e ilustram diferenças descritivas. **Atenção:** 23 linhas positivas correspondem a 1 contrato; 100 negativas a 100 contratos. O gráfico PCA é uma projeção geométrica, não uma rede de transações, não é generalização e não implica agrupamentos de atividades ilícitas.

In [ ]:
D=pd.read_parquet(ROOT/'features_treino_original.parquet')
features=['n_tx','value_entropy','fee_entropy','internal_value_entropy','call_count_mean']
assert set(features+['label','recipient']).issubset(D.columns)
assert D.loc[D.label==1,'recipient'].nunique()==1
for feature in ['value_entropy','fee_entropy','internal_value_entropy','call_count_mean']:
   data=[D.loc[D.label==0,feature].astype(float).to_numpy(),D.loc[D.label==1,feature].astype(float).to_numpy()]
   fig,ax=plt.subplots(figsize=(6.5,3.9));parts=ax.boxplot(data,tick_labels=['100 negativos\n(100 contratos)','23 positivos\n(1 contrato)'],patch_artist=True,showfliers=True)
   for patch,color in zip(parts['boxes'],[GRAY,PETROL]):patch.set_facecolor(color);patch.set_alpha(.55)
   ax.set_title('Distribuição de '+feature);ax.set_ylabel(feature)
   export('04_boxplot_'+feature+'.png')
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
X=StandardScaler().fit_transform(D[features].astype(float).replace([np.inf,-np.inf],np.nan).fillna(D[features].median()))
proj=PCA(n_components=2).fit_transform(X)
fig,ax=plt.subplots(figsize=(7.8,5.5))
for lab,col,title in [(0,GRAY,'Classe 0: 100 contratos'),(1,PETROL,'Classe 1: 1 contrato/23 dias')]:
 mask=D.label.to_numpy()==lab;ax.scatter(proj[mask,0],proj[mask,1],label=title,s=58 if lab else 28,alpha=.8,c=col,marker='D' if lab else 'o')
ax.legend(loc='best',fontsize=9);ax.set(xlabel='Componente principal 1',ylabel='Componente principal 2',title='PCA das 123 observações de treino — visão exploratória')
export('05_pca_treino_exploratoria.png')

## 5. Visualização dos alertas de janeiro de 2026

As nove pontuações são do arquivo `sinalizados_teste_2026.csv`. Os seis contratos sem rótulo de referência são **indeterminados**, não negativos. O scatter plot é de **atributos estatísticos**, e não um grafo real de transferências.

In [ ]:
A=pd.read_csv(ROOT/'sinalizados_teste_2026.csv')
assert len(A)==9
is_ref=A['rotulo'].fillna('').isin(['0.1 ETH','1 ETH','10 ETH'])
assert int(is_ref.sum())==3
Z=A.sort_values('prob',ascending=True).copy();ref=Z['rotulo'].fillna('').isin(['0.1 ETH','1 ETH','10 ETH'])
fig,ax=plt.subplots(figsize=(8,5.3));y=np.arange(len(Z))
ax.hlines(y,0,Z.prob,color=np.where(ref,PETROL,GRAY),lw=3,alpha=.7)
ax.scatter(Z.prob,y,c=np.where(ref,PETROL,GRAY),s=95,zorder=3)
ax.set_yticks(y,[f'{"Tornado Cash" if r else "Indeterminado"} · {a[:8]}…' for a,r in zip(Z.recipient,ref)])
ax.set_xlim(.0,1.1);ax.set(xlabel='Pontuação produzida pelo classificador (não é prova)',title='Janeiro/2026 — escores dos nove alertas')
ax.axvline(.85,ls='--',color=ORANGE,label='Limiar congelado 0,85');ax.legend(fontsize=9)
export('06_alertas_2026_lollipop.png')
fig,ax=plt.subplots(figsize=(8,5.2))
for v,col,lab in [(True,PETROL,'Três contratos conhecidos'),(False,GRAY,'Seis indeterminados')]:
 m=is_ref==v;ax.scatter(A.loc[m,'fee_entropy'],A.loc[m,'call_count_mean'],s=35+2*A.loc[m,'n_tx'],alpha=.78,c=col,label=lab,edgecolors='white',linewidths=.65)
ax.set_yscale('log');ax.set(xlabel='fee_entropy',ylabel='call_count_mean (escala log)',title='Atributos dos alertas — visualização exploratória')
ax.legend(fontsize=9);export('07_dispersao_atributos_alertas.png')

## 6. Matriz de confusão agregada dos cinco folds (EXP-07)

Os registros por fold já foram auditados. A agregação a seguir soma as predições internas, sem tratá-las como evidência de generalização. Esta visualização não é a matriz de confusão da população Ethereum.

In [ ]:
F=pd.read_csv(ROOT/'metricas_12_combinacoes_por_fold.csv')
r=F[(F.configuracao=='A_original_5')&(F.modelo=='RandomForest')]
assert len(r)==5
cm=np.array([[r.tn.sum(),r.fp.sum()],[r.fn.sum(),r.tp.sum()]],dtype=int)
fig,ax=plt.subplots(figsize=(6,4.5));im=ax.imshow(cm,cmap='Blues')
ax.set(xticks=[0,1],yticks=[0,1],xticklabels=['Classe 0','Classe 1'],yticklabels=['Classe 0','Classe 1'],xlabel='Predito',ylabel='Rótulo da matriz',title='Confusão agregada — 5 folds internos')
for i in range(2):
 for j in range(2):ax.text(j,i,str(cm[i,j]),ha='center',va='center',fontweight='bold',color='white' if cm[i,j]>cm.max()/2 else 'black',fontsize=18)
fig.colorbar(im,ax=ax);export('08_confusao_interna_rf.png')
print('AVISO: contrato positivo é o mesmo nos cinco folds. NUNCA apresentar como teste em novo protocolo.')

## 7. Opcional — reexecutar 12 classificadores com a matriz real

Não é necessário para gerar as figuras. Para auditoria dos modelos, execute [EXP-07 v2 original](https://drive.google.com/file/d/1nKI1sM5pbY-UOAI6nxraTTSzkNFY8MHW/view), que contém todo o protocolo pré-registrado, hashes e gravação em Drive. O notebook integrado é o **painel de estudo e apresentação**; o EXP-07 v2 original continua como registro autoritativo de execução.

## 8. Exportar gráficos para seu Drive (opcional)

`EXPORTAR_PARA_DRIVE=False` por padrão: a demonstração é segura contra criação de pastas duplicadas. Ao ativar, o Colab cria uma pasta privada no Drive e envia os PNGs. Os gráficos continuam na pasta local `/content/lab_ethereum/figures/` mesmo sem exportação.

In [ ]:
EXPORTAR_PARA_DRIVE=False
if EXPORTAR_PARA_DRIVE:
    name='LAB_GRAFICOS_'+datetime.now(timezone.utc).strftime('%Y%m%dT%H%M%SZ')
    folder=service.files().create(body={'name':name,'mimeType':'application/vnd.google-apps.folder','parents':['1iZKAeZPEb6JOlH8KfqsR6-wqI1ygmojI']},fields='id,name').execute()
    for p in sorted(FIG.glob('*.png')):
        media=MediaFileUpload(str(p),mimetype='image/png',resumable=True)
        service.files().create(body={'name':p.name,'parents':[folder['id']]},media_body=media,fields='id').execute()
    print('Gráficos salvos no Drive na pasta de resultados:',folder['name'],folder['id'])
else:print('Gráficos disponíveis para inspeção neste Colab:',FIG)